# Model Evaluation and Selection Lab

Comprehensive guide to evaluating, comparing, and selecting machine learning models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     learning_curve, GridSearchCV, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score, classification_report,
                             calibration_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy import stats
np.random.seed(42)
%matplotlib inline

## 1. Train/Test Split - Why It Matters

In [ ]:
# Generate data
X, y = make_classification(n_samples=300, n_features=20, n_informative=5,
                           n_redundant=5, random_state=42)

# Without split - overfitting not detected
tree_overfit = DecisionTreeClassifier(random_state=42)
tree_overfit.fit(X, y)
print(f'NO SPLIT - Training accuracy: {tree_overfit.score(X, y):.4f}')
print('Looks perfect! But is it really?\n')

# With split - see true performance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
tree_overfit.fit(X_train, y_train)
print(f'WITH SPLIT:')
print(f'  Training accuracy: {tree_overfit.score(X_train, y_train):.4f}')
print(f'  Test accuracy:     {tree_overfit.score(X_test, y_test):.4f}')
print(f'  Gap reveals overfitting!')

## 2. K-Fold Cross-Validation

In [ ]:
# Manual K-Fold implementation
def manual_kfold(X, y, model, k=5):
    n = len(X)
    fold_size = n // k
    indices = np.random.permutation(n)
    scores = []
    
    for i in range(k):
        val_idx = indices[i*fold_size:(i+1)*fold_size]
        train_idx = np.concatenate([indices[:i*fold_size], indices[(i+1)*fold_size:]])
        
        model.fit(X[train_idx], y[train_idx])
        score = model.score(X[val_idx], y[val_idx])
        scores.append(score)
    
    return np.array(scores)

# Compare with sklearn
model = LogisticRegression(max_iter=1000)
manual_scores = manual_kfold(X, y, model, k=5)
sklearn_scores = cross_val_score(model, X, y, cv=5)

print(f'Manual K-Fold:  {manual_scores.mean():.4f} (+/- {manual_scores.std():.4f})')
print(f'Sklearn K-Fold: {sklearn_scores.mean():.4f} (+/- {sklearn_scores.std():.4f})')

In [ ]:
# Visualize K-Fold splits
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fig, ax = plt.subplots(figsize=(12, 4))

for i, (train_idx, val_idx) in enumerate(kf.split(X)):
    ax.scatter(train_idx, [i]*len(train_idx), c='blue', s=1, alpha=0.3)
    ax.scatter(val_idx, [i]*len(val_idx), c='red', s=3)

ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Sample Index')
ax.set_title('K-Fold Cross-Validation Splits (red=validation, blue=training)')
plt.tight_layout()
plt.show()

## 3. Confusion Matrix

In [ ]:
# Train a model and show confusion matrix
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

lr = LogisticRegression(max_iter=5000)
lr.fit(X_tr_s, y_tr)
y_pred = lr.predict(X_te_s)

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Malignant', 'Benign'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - Breast Cancer')
plt.show()

print(classification_report(y_te, y_pred, target_names=['Malignant', 'Benign']))

## 4. ROC Curve and AUC

In [ ]:
# ROC curves for multiple models
models_roc = {
    'Logistic Regression': LogisticRegression(max_iter=5000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
}

fig, ax = plt.subplots(figsize=(8, 7))
for name, model in models_roc.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_tr, y_tr)
    if hasattr(pipe, 'predict_proba'):
        y_score = pipe.predict_proba(X_te)[:, 1]
    else:
        y_score = pipe.decision_function(X_te)
    fpr, tpr, _ = roc_curve(y_te, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Multiple Models')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.show()

## 5. Precision-Recall Curve (Imbalanced Data)

In [ ]:
# Create imbalanced dataset
X_imb, y_imb = make_classification(n_samples=1000, n_features=20, weights=[0.9, 0.1],
                                   n_informative=5, random_state=42)
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)
print(f'Class distribution: {np.bincount(y_te_i)} (highly imbalanced)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, model in [('Logistic Regression', LogisticRegression(max_iter=5000)),
                    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42))]:
    model.fit(X_tr_i, y_tr_i)
    y_score = model.predict_proba(X_te_i)[:, 1]
    
    # ROC
    fpr, tpr, _ = roc_curve(y_te_i, y_score)
    axes[0].plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    
    # PR curve
    prec, rec, _ = precision_recall_curve(y_te_i, y_score)
    ap = average_precision_score(y_te_i, y_score)
    axes[1].plot(rec, prec, linewidth=2, label=f'{name} (AP={ap:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curve (can be misleading for imbalanced data)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

axes[1].axhline(y=y_te_i.mean(), color='k', linestyle='--', label='Baseline')
axes[1].set_title('Precision-Recall Curve (better for imbalanced)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Bias-Variance Tradeoff

In [ ]:
# Vary model complexity and observe bias-variance tradeoff
complexities = range(1, 25)
train_scores_bv = []
test_scores_bv = []

for depth in complexities:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores_bv.append(tree.score(X_train, y_train))
    test_scores_bv.append(tree.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(complexities, train_scores_bv, 'b-o', label='Training Score', markersize=4)
plt.plot(complexities, test_scores_bv, 'r-o', label='Test Score', markersize=4)
plt.fill_between(complexities, test_scores_bv, train_scores_bv, alpha=0.1, color='gray', label='Generalization Gap')
plt.xlabel('Model Complexity (max_depth)')
plt.ylabel('Accuracy')
plt.title('Bias-Variance Tradeoff')
plt.legend()
plt.grid(True, alpha=0.3)
plt.annotate('Underfitting\n(High Bias)', xy=(2, 0.75), fontsize=12)
plt.annotate('Overfitting\n(High Variance)', xy=(18, 0.85), fontsize=12)
plt.show()

## 7. Learning Curves (Training Size vs Score)

In [ ]:
# Learning curves for different models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_lc = [
    ('Logistic Regression', LogisticRegression(max_iter=5000)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=50, random_state=42)),
]

for ax, (name, model) in zip(axes, models_lc):
    train_sizes, train_sc, val_sc = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy')
    ax.plot(train_sizes, train_sc.mean(axis=1), 'b-o', label='Train', markersize=4)
    ax.plot(train_sizes, val_sc.mean(axis=1), 'r-o', label='Val', markersize=4)
    ax.fill_between(train_sizes, train_sc.mean(1)-train_sc.std(1), train_sc.mean(1)+train_sc.std(1), alpha=0.1, color='b')
    ax.fill_between(train_sizes, val_sc.mean(1)-val_sc.std(1), val_sc.mean(1)+val_sc.std(1), alpha=0.1, color='r')
    ax.set_title(name); ax.set_xlabel('Training Size'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.set_ylim(0.6, 1.05)
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning: Grid vs Random

In [ ]:
# Grid Search
param_grid = {'max_depth': [3, 5, 7, 10, 15], 'n_estimators': [10, 50, 100, 200],
              'min_samples_split': [2, 5, 10]}

start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
grid_time = time.time() - start
print(f'Grid Search: {grid.best_score_:.4f} in {grid_time:.2f}s ({grid.cv_results_["mean_test_score"].shape[0]} fits)')
print(f'  Best params: {grid.best_params_}')

# Random Search (same budget)
from scipy.stats import randint
param_dist = {'max_depth': randint(2, 20), 'n_estimators': randint(10, 200),
              'min_samples_split': randint(2, 20)}

start = time.time()
random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, 
                                   n_iter=60, cv=5, scoring='accuracy', random_state=42)
random_search.fit(X_train, y_train)
rand_time = time.time() - start
print(f'\nRandom Search: {random_search.best_score_:.4f} in {rand_time:.2f}s (60 fits)')
print(f'  Best params: {random_search.best_params_}')
print(f'\nRandom search explores more diverse combinations with fewer evaluations')

import time

## 9. Calibration Curves

In [ ]:
# Calibration curves - how well do predicted probabilities match reality?
from sklearn.calibration import CalibratedClassifierCV

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')

for name, model in [('Logistic Regression', LogisticRegression(max_iter=5000)),
                    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
                    ('SVM (Platt)', SVC(probability=True, random_state=42))]:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_tr, y_tr)
    prob_pos = pipe.predict_proba(X_te)[:, 1]
    fraction_of_positives, mean_predicted_value = calibration_curve(y_te, prob_pos, n_bins=10)
    ax.plot(mean_predicted_value, fraction_of_positives, 's-', label=name)

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.show()
print('A well-calibrated model: predicted probability matches actual frequency')

## 10. Statistical Comparison of Models

In [ ]:
# Paired t-test on cross-validation scores
models_compare = {
    'Logistic Regression': LogisticRegression(max_iter=5000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Use same CV folds for fair comparison
cv = KFold(n_splits=10, shuffle=True, random_state=42)
all_scores = {}
for name, model in models_compare.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    all_scores[name] = scores
    print(f'{name:25s}: {scores.mean():.4f} +/- {scores.std():.4f}')

# Paired t-test between top two
print('\n--- Statistical Tests (paired t-test on CV folds) ---')
names = list(all_scores.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        t_stat, p_val = stats.ttest_rel(all_scores[names[i]], all_scores[names[j]])
        sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else 'ns'
        print(f'{names[i]} vs {names[j]}: t={t_stat:.3f}, p={p_val:.4f} [{sig}]')

In [ ]:
# Visualize CV score distributions
fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(all_scores))
bp = ax.boxplot(all_scores.values(), positions=positions, widths=0.6, patch_artist=True)
colors = ['lightblue', 'lightgreen', 'lightsalmon']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xticks(positions)
ax.set_xticklabels(all_scores.keys())
ax.set_ylabel('Accuracy')
ax.set_title('Cross-Validation Score Distributions')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Summary

**Key Takeaways:**
1. Always use train/test split (or CV) to estimate generalization
2. ROC-AUC for balanced data, PR-AUC for imbalanced
3. Learning curves diagnose underfitting vs overfitting
4. Random search is often more efficient than grid search
5. Use statistical tests to confirm if differences are significant
6. Check calibration if you need reliable probability estimates